## Probar con DOC2
## Se reemplaza la imagen de curva por la imagen de OBRAS y se usa la imagen de CRUCE de color naranja

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Carpeta de imagenes sin fondo
carpeta='img/sinfondo'
imagenes=sorted(glob.glob(os.path.join(carpeta,'*.png')))
# Quita imagen anterior de cruce y deja la nueva crucenar
imagenes=[r for r in imagenes if ('cruce' not in os.path.basename(r).lower()) or ('crucenar' in os.path.basename(r).lower())]

# Rangos HSV ajustados (igual que Detectar_colores)
azul_b=np.array([85,70,60],np.uint8)
azul_a=np.array([135,255,255],np.uint8)
amarillo_b=np.array([18,70,70],np.uint8)
amarillo_a=np.array([40,255,255],np.uint8)
rojo_b1=np.array([0,100,70],np.uint8)
rojo_a1=np.array([10,255,255],np.uint8)
rojo_b2=np.array([170,100,70],np.uint8)
rojo_a2=np.array([180,255,255],np.uint8)

# Kernel y umbral de area
kernel=np.ones((3,3),np.uint8)
min_area=120

for ruta in imagenes:
    img=cv2.imread(ruta,cv2.IMREAD_UNCHANGED)
    if img is None:
        print('No se pudo leer:',ruta)
        continue

    # Si tiene alpha, se procesa en BGR manteniendo el objeto principal
    if len(img.shape)==3 and img.shape[2]==4:
        bgr=img[:,:,:3].copy()
        alpha=img[:,:,3]
    else:
        bgr=img.copy()
        alpha=None

    bgr=cv2.resize(bgr,(500,500))
    frame_s=cv2.GaussianBlur(bgr,(7,7),0)
    hsv=cv2.cvtColor(frame_s,cv2.COLOR_BGR2HSV)

    # Mascaras por color
    mask_azul=cv2.inRange(hsv,azul_b,azul_a)
    mask_amarillo=cv2.inRange(hsv,amarillo_b,amarillo_a)
    mask_rojo=cv2.bitwise_or(cv2.inRange(hsv,rojo_b1,rojo_a1),cv2.inRange(hsv,rojo_b2,rojo_a2))

    # Si habia alpha, limitar deteccion al objeto
    if alpha is not None:
        alpha=cv2.resize(alpha,(500,500))
        _,mask_obj=cv2.threshold(alpha,1,255,cv2.THRESH_BINARY)
        mask_azul=cv2.bitwise_and(mask_azul,mask_obj)
        mask_amarillo=cv2.bitwise_and(mask_amarillo,mask_obj)
        mask_rojo=cv2.bitwise_and(mask_rojo,mask_obj)

    # Limpieza morfologica
    mask_azul=cv2.morphologyEx(mask_azul,cv2.MORPH_OPEN,kernel)
    mask_azul=cv2.morphologyEx(mask_azul,cv2.MORPH_CLOSE,kernel)
    mask_amarillo=cv2.morphologyEx(mask_amarillo,cv2.MORPH_OPEN,kernel)
    mask_amarillo=cv2.morphologyEx(mask_amarillo,cv2.MORPH_CLOSE,kernel)
    mask_rojo=cv2.morphologyEx(mask_rojo,cv2.MORPH_OPEN,kernel)
    mask_rojo=cv2.morphologyEx(mask_rojo,cv2.MORPH_CLOSE,kernel)

    # Contornos por color
    cont_azul,_=cv2.findContours(mask_azul,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cont_amarillo,_=cv2.findContours(mask_amarillo,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cont_rojo,_=cv2.findContours(mask_rojo,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    salida=bgr.copy()

    # Azul -> contorno rojo
    for c in cont_azul:
        area=cv2.contourArea(c)
        if area>min_area:
            cv2.drawContours(salida,[c],-1,(0,0,255),2)

    # Rojo -> contorno amarillo
    for c in cont_rojo:
        area=cv2.contourArea(c)
        if area>min_area:
            cv2.drawContours(salida,[c],-1,(0,255,255),2)

    # Amarillo -> contorno azul
    for c in cont_amarillo:
        area=cv2.contourArea(c)
        if area>min_area:
            cv2.drawContours(salida,[c],-1,(255,0,0),2)

    # Visualizacion
    nombre=os.path.basename(ruta)
    bgr_rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    salida_rgb=cv2.cvtColor(salida,cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(bgr_rgb,vmin=0,vmax=255)
    plt.title(f'Original: {nombre}')
    plt.xticks([])
    plt.yticks([])

    plt.subplot(1,3,2)
    mezcla=cv2.add(mask_rojo,cv2.add(mask_amarillo,mask_azul))
    plt.imshow(mezcla,vmin=0,vmax=255,cmap='gray')
    plt.title('Mascara combinada RGB')
    plt.xticks([])
    plt.yticks([])

    plt.subplot(1,3,3)
    plt.imshow(salida_rgb,vmin=0,vmax=255)
    plt.title('Contornos por color')
    plt.xticks([])
    plt.yticks([])

    plt.show()

    print(nombre,'-> azul:',len(cont_azul),'rojo:',len(cont_rojo),'amarillo:',len(cont_amarillo))

In [ ]:
import numpy as np
import cv2

# Deteccion en tiempo real por camara con clasificacion de senales
cam=cv2.VideoCapture(0)

if not cam.isOpened():
    print('No se pudo abrir la camara')
else:
    # Rangos HSV
    azul_b=np.array([85,70,60],np.uint8)
    azul_a=np.array([135,255,255],np.uint8)

    # Naranja para CRUCE
    naranja_b=np.array([6,90,80],np.uint8)
    naranja_a=np.array([22,255,255],np.uint8)

    rojo_b1=np.array([0,100,70],np.uint8)
    rojo_a1=np.array([10,255,255],np.uint8)
    rojo_b2=np.array([170,100,70],np.uint8)
    rojo_a2=np.array([180,255,255],np.uint8)

    verde_b=np.array([35,60,60],np.uint8)
    verde_a=np.array([90,255,255],np.uint8)

    kernel=np.ones((3,3),np.uint8)
    min_area=450

    while True:
        ret,frame=cam.read()
        if not ret:
            break

        h_frame,w_frame=frame.shape[:2]

        # Cuadro central de deteccion (solo esta zona)
        x1=int(w_frame*0.25)
        y1=int(h_frame*0.20)
        x2=int(w_frame*0.75)
        y2=int(h_frame*0.80)

        roi=frame[y1:y2,x1:x2].copy()

        frame_s=cv2.GaussianBlur(roi,(7,7),0)
        hsv=cv2.cvtColor(frame_s,cv2.COLOR_BGR2HSV)

        # Mascaras por color
        mask_azul=cv2.inRange(hsv,azul_b,azul_a)
        mask_naranja=cv2.inRange(hsv,naranja_b,naranja_a)
        mask_rojo=cv2.bitwise_or(cv2.inRange(hsv,rojo_b1,rojo_a1),cv2.inRange(hsv,rojo_b2,rojo_a2))
        mask_verde=cv2.inRange(hsv,verde_b,verde_a)

        # Limpieza morfologica
        mask_azul=cv2.morphologyEx(mask_azul,cv2.MORPH_OPEN,kernel)
        mask_azul=cv2.morphologyEx(mask_azul,cv2.MORPH_CLOSE,kernel)

        mask_naranja=cv2.morphologyEx(mask_naranja,cv2.MORPH_OPEN,kernel)
        mask_naranja=cv2.morphologyEx(mask_naranja,cv2.MORPH_CLOSE,kernel)

        mask_rojo=cv2.morphologyEx(mask_rojo,cv2.MORPH_OPEN,kernel)
        mask_rojo=cv2.morphologyEx(mask_rojo,cv2.MORPH_CLOSE,kernel)

        mask_verde=cv2.morphologyEx(mask_verde,cv2.MORPH_OPEN,kernel)
        mask_verde=cv2.morphologyEx(mask_verde,cv2.MORPH_CLOSE,kernel)

        # Unir rectangulos verdes cercanos (para no detectar 3 destinos separados)
        mask_verde_union=cv2.dilate(mask_verde,np.ones((9,9),np.uint8),iterations=1)
        mask_verde_union=cv2.morphologyEx(mask_verde_union,cv2.MORPH_CLOSE,np.ones((11,11),np.uint8))

        # Buscar contornos
        cont_azul,_=cv2.findContours(mask_azul,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cont_naranja,_=cv2.findContours(mask_naranja,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cont_rojo,_=cv2.findContours(mask_rojo,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cont_verde,_=cv2.findContours(mask_verde_union,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

        # 1) AZUL -> AUTOBUS (mostrar una sola vez)
        mejor_azul=None
        area_azul_max=0
        for c in cont_azul:
            area=cv2.contourArea(c)
            if area>min_area and area>area_azul_max:
                area_azul_max=area
                mejor_azul=c

        if mejor_azul is not None:
            x,y,w,h=cv2.boundingRect(mejor_azul)
            precision=min(99,int(60+(area_azul_max/1800)))
            cv2.drawContours(roi,[mejor_azul],-1,(0,0,255),3)
            cv2.putText(roi,'AUTOBUS '+str(precision)+'%',(x,y-8),cv2.FONT_HERSHEY_SIMPLEX,0.62,(0,0,255),2)

        # 2) ROJO -> ALTO / CEDA EL PASO / OBRAS / 20KMH
        for c in cont_rojo:
            area=cv2.contourArea(c)
            if area<=min_area:
                continue

            per=cv2.arcLength(c,True)
            if per<=0:
                continue

            ap=cv2.approxPolyDP(c,0.02*per,True)
            lados=len(ap)
            x,y,w,h=cv2.boundingRect(c)
            asp=w/float(h) if h>0 else 0.0
            circ=(4*np.pi*area)/(per*per)

            _,radio=cv2.minEnclosingCircle(c)
            fill_circ=0.0
            if radio>0:
                fill_circ=area/(np.pi*radio*radio)

            etiqueta=''
            precision=0

            # ALTO primero para evitar confundirlo con 20KMH
            if 6<=lados<=10 and 0.65<asp<1.35 and 0.42<circ<0.90 and fill_circ<0.94:
                etiqueta='ALTO'
                p_lados=100-abs(8-lados)*8
                p_circ=int(min(100,max(0,(0.90-circ)*220)))
                precision=max(70,min(99,int((p_lados+p_circ)/2)))

            # 20KMH solo si es claramente circular
            elif circ>0.90 and 0.92<asp<1.08 and fill_circ>0.93 and lados>=10:
                etiqueta='20KMH'
                precision=int(min(99,(circ*55)+(fill_circ*45)))

            # Triangulos rojos: CEDA (invertido) y OBRAS (normal)
            elif lados==3:
                pts=ap.reshape(-1,2)
                ys=np.sort(pts[:,1])

                # CEDA: base arriba y punta abajo (triangulo invertido)
                if abs(int(ys[0])-int(ys[1])) < (0.20*h) and abs(int(ys[2])-int(ys[1])) > (0.25*h):
                    etiqueta='CEDA EL PASO'
                    precision=90

                # OBRAS: base abajo y punta arriba (triangulo normal)
                elif abs(int(ys[2])-int(ys[1])) < (0.20*h) and abs(int(ys[1])-int(ys[0])) > (0.25*h):
                    etiqueta='OBRAS'
                    precision=90

            cv2.drawContours(roi,[c],-1,(0,255,255),3)
            if etiqueta!='':
                cv2.putText(roi,etiqueta+' '+str(precision)+'%',(x,y-8),cv2.FONT_HERSHEY_SIMPLEX,0.58,(0,255,255),2)

        # 3) NARANJA (ROMBO) -> CRUCE
        mejor_naranja=None
        area_naranja_max=0
        for c in cont_naranja:
            area=cv2.contourArea(c)
            if area<=min_area:
                continue
            per=cv2.arcLength(c,True)
            if per<=0:
                continue
            ap=cv2.approxPolyDP(c,0.02*per,True)
            lados=len(ap)
            x,y,w,h=cv2.boundingRect(c)
            asp=w/float(h) if h>0 else 0.0
            if lados==4 and 0.70<asp<1.35 and area>area_naranja_max:
                area_naranja_max=area
                mejor_naranja=c

        if mejor_naranja is not None:
            x,y,w,h=cv2.boundingRect(mejor_naranja)
            precision=min(99,int(72+(area_naranja_max/2300)))
            cv2.drawContours(roi,[mejor_naranja],-1,(255,0,0),3)
            cv2.putText(roi,'CRUCE '+str(precision)+'%',(x,y-8),cv2.FONT_HERSHEY_SIMPLEX,0.58,(255,0,0),2)

        # 4) VERDE -> DESTINO (mostrar una sola vez, agrupado)
        mejor_verde=None
        area_verde_max=0
        for c in cont_verde:
            area=cv2.contourArea(c)
            if area>min_area and area>area_verde_max:
                area_verde_max=area
                mejor_verde=c

        if mejor_verde is not None:
            x,y,w,h=cv2.boundingRect(mejor_verde)
            m=8
            x=max(0,x-m)
            y=max(0,y-m)
            w=min(roi.shape[1]-x,w+2*m)
            h=min(roi.shape[0]-y,h+2*m)
            precision=min(99,int(65+(area_verde_max/2200)))
            cv2.rectangle(roi,(x,y),(x+w,y+h),(0,255,0),3)
            cv2.putText(roi,'DESTINO '+str(precision)+'%',(x,y-8),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)

        # Pegar ROI procesada y dibujar cuadro guia
        frame[y1:y2,x1:x2]=roi
        cv2.rectangle(frame,(x1,y1),(x2,y2),(255,255,0),2)

        cv2.imshow('Deteccion por camara - ProyectoExitoso',frame)

        # Presiona 0 para salir
        if cv2.waitKey(1)&0xFF==ord('0'):
            break

cam.release()
cv2.destroyAllWindows()